# Spelling Correction System
## CT052-3-M-NLP Assignment - Part A1

**Students:**
- TP139069 - Muhammad Mahdi Hardi Effendi
- TP131138 - Alieu Bojang

This system implements the **Noisy Channel Model** for spelling correction as taught in the lecture slides.

**Core Formula:** ŵ = argmax P(x|w) × P(w)

The system corrects:
- **Non-word errors**: misspelled words not in the dictionary (e.g., *acress* → *actress*)
- **Real-word errors**: valid words used in the wrong context (e.g., *thew* → *the*)

**Techniques Used:**
- **Corpus**: Reuters (business/finance domain, ~1.7M words)
- **Edit Distance** (Damerau-Levenshtein): candidate generation (4 operations: insert, delete, substitute, transpose)
- **Confusion Matrices**: channel model P(x|w) — probability of a typo given the intended word
- **Bigram Language Model** with Laplace smoothing: prior P(w) and sentence scoring for real-word errors
- **Noisy Channel Model**: combines channel model and language model to rank candidates

## Step 1: Setup and Load Corpus

In [1]:
import nltk
import re
import math
from collections import Counter, defaultdict

# Download required data (run once)
nltk.download('brown', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.corpus import brown

# NLTK's frequency distribution class
from nltk.probability import FreqDist

# NLTK's Laplace-smoothed language model
from nltk.lm import Laplace as LaplaceLM

# Preprocessing for LM
from nltk.lm.preprocessing import padded_everygram_pipeline
     

# Load all words from brown corpus
raw_words = brown.words()
print(f"Total word tokens in Brown corpus: {len(raw_words)}")
print(f"First 20 words: {list(raw_words[:20])}")

Total word tokens in Brown corpus: 1161192
First 20 words: ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that']


In [2]:
# from nltk.corpus import reuters

# # Load all of the word from the reuters corpus
# # reuters.words() will return a list of every token in the corpus
# raw_words = reuters.words()
# print(f"Total word tokens in Reuters Corpus: {len(raw_words)}")
# print(f"First 20 words: {list(raw_words[:20])}")

In [3]:
len(set(raw_words))

56057

## Step 2: Build Vocabulary and Word Frequencies

We need two things from the corpus:
1. **Vocabulary** — the set of all correctly spelled words (our dictionary)
2. **Word frequencies** — how often each word appears

The word frequency gives us the **Prior / Language Model P(w)**:

P(w) = count(w) / N

where N = total number of words in the corpus.

In [4]:
# Clean the corpus: lowercase, keep only the alphabetic words, 
# and filter out numbers and punctuation - that are not "words" for spell checking
corpus_words = [w.lower() for w in raw_words if w.isalpha()]

print(f"Total clean word tokens: {len(corpus_words)}")
print(f"Total Unique clean words: {len(set(corpus_words))}")
print(f"First 10 clean words: {corpus_words[:10]}")

Total clean word tokens: 981716
Total Unique clean words: 40234
First 10 clean words: ['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of']


In [5]:
# Count word frequencies using NLTK's FreqDist

# FreqDist is NLTK's built-in frequency distribution class.
# It works like Counter but adds useful NLP methods:
#   .freq(word) — returns the relative frequency (probability)
#   .most_common(n) — top n most frequent words
#   .N() — total number of tokens
#   .B() — number of unique types (vocabulary size)

word_counts = FreqDist(corpus_words)

# The vocabulary = all unique words in the corpus
VOCABULARY = set(word_counts.keys())

# Total number of word tokens (FreqDist provides .N() for this)
TOTAL_WORDS = word_counts.N()

print(f"Vocabulary size (unique words): {word_counts.B()}")
print(f"Total word tokens: {word_counts.N()}")
print(f"\nTop 10 most common words:")
for word, count in word_counts.most_common(10):
    # .freq(word) gives the relative frequency = count / N
    print(f"  {word}: {count} times (P = {word_counts.freq(word):.6f})")

Vocabulary size (unique words): 40234
Total word tokens: 981716

Top 10 most common words:
  the: 69971 times (P = 0.071274)
  of: 36412 times (P = 0.037090)
  and: 28853 times (P = 0.029390)
  to: 26158 times (P = 0.026645)
  a: 23195 times (P = 0.023627)
  in: 21337 times (P = 0.021734)
  that: 10594 times (P = 0.010791)
  is: 10109 times (P = 0.010297)
  was: 9815 times (P = 0.009998)
  he: 9548 times (P = 0.009726)


In [6]:
def word_probability(word):
    """
    Unigram Language Model with Laplace smoothing:
    P(word) = (count(word) + 1) / (N + |V|)
    
    Uses FreqDist's word_counts[word] to get the count.
    Laplace smoothing adds 1 to numerator and |V| to denominator
    so that unknown words get a small probability instead of 0.
    """
    return (word_counts[word] + 1) / (word_counts.N() + word_counts.B())

# Testing the function
print(f"P('the') = {word_probability('the'):.6f})")
print(f"P('market') = {word_probability('market'):.6f})")
print(f"P('xyzbank') = {word_probability('xyzbank'):.10f}) -> the unknow word, still get probability but very small instead of 0")

P('the') = 0.068469)
P('market') = 0.000153)
P('xyzbank') = 0.0000009785) -> the unknow word, still get probability but very small instead of 0


## Step 3: Build Bigram Language Model (NLTK)

For **real-word error correction**, we need to know which words fit well together in context.

A bigram model gives us P(word₂ | word₁) — the probability of seeing word₂ after word₁.

**Formula (MLE with Laplace smoothing):**

P(wₙ | wₙ₋₁) = (count(wₙ₋₁, wₙ) + 1) / (count(wₙ₋₁) + |V|)

This is used to score entire sentences:

P(W) = P(w₁|\<s\>) × P(w₂|w₁) × P(w₃|w₂) × ...

We use **NLTK's `Laplace` language model** (from Lab 08) with `padded_everygram_pipeline` to build
the bigram model. This handles:
- Padding sentences with `<s>` and `</s>` markers automatically
- Laplace smoothing for unseen bigrams
- Scoring with `.score(word, context)` and `.entropy()` / `.perplexity()`

In [7]:
# Build Bigram Language Model using NLTK (Lab 08 approach)

# padded_everygram_pipeline(n, tokenized_text):
#   - Pads each sentence with <s> (start) and </s> (end) markers
#   - Generates n-grams (unigrams + bigrams for n=2)
#   - Returns (train_data, vocab) ready for the LM
#
# Laplace(order=2):
#   - Creates a bigram model with Laplace (add-1) smoothing
#   - .fit(train_data, vocab) trains it on the corpus
#   - .score(word, context) gives P(word | context)

# First is to Tokenize corpus into list of sentences (list of word lists)
# Brown corpus has provides the .sents() which returns pre-tokenized sentences

tokenized_sents = []
for sent in brown.sents():
    # Clean: lowercase, keep only alphabetic words
    clean_sent = [w.lower() for w in sent if w.isalpha()]
    if clean_sent:  # skip empty sentences
        tokenized_sents.append(clean_sent)

print(f"Total sentences for LM training: {len(tokenized_sents)}")
print(f"Example sentence: {tokenized_sents[0]}")

# Use padded_everygram_pipeline to prepare training data
# n=2 means bigram model (considers pairs of words)
n = 2  # bigram
train_data, vocab = padded_everygram_pipeline(n, tokenized_sents)

# Create and train the Laplace-smoothed bigram model
bigram_model = LaplaceLM(order=n)
bigram_model.fit(train_data, vocab)

print(f"\nBigram Language Model trained successfully!")
print(f"Vocabulary size in LM: {len(bigram_model.vocab)}")

# Test the model — score individual bigrams
# .score(word, context) gives P(word | context)
# context must be a list of previous words
print(f"\nBigram probabilities:")
print(f"  P('world' | 'the') = {bigram_model.score('world', ['the']):.6f}  <-- high")
print(f"  P('banana' | 'the') = {bigram_model.score('banana', ['the']):.6f}  <-- low")
print(f"  P('the'   | '<s>')   = {bigram_model.score('the', ['<s>']):.6f}  <-- sentence start")

Total sentences for LM training: 56766
Example sentence: ['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place']

Bigram Language Model trained successfully!
Vocabulary size in LM: 40237

Bigram probabilities:
  P('world' | 'the') = 0.003412  <-- high
  P('banana' | 'the') = 0.000009  <-- low
  P('the'   | '<s>')   = 0.073338  <-- sentence start


In [8]:
def bigram_probability(word, prev_word):
    """
    Wrapper around the NLTK's bigram_model.score().

    Returns P(word | prev_word) using the Laplace-smoothed bigram model.

    bigram_model.score(word, context):
        - word: the word we want the probability of
        - context: list of previous words (for bigram or just [prev_word])
        - Returns the laplace-smoothed conditional probability
    """
    return bigram_model.score(word, [prev_word])

# Testing the function
print(f"P('world' | 'the') = {bigram_probability('world', 'the')}")
print(f"P('are' | 'how') = {bigram_probability('are', 'how')}")
print(f"P('Alieu' | 'Mahdi') = {bigram_probability('Alieu', 'Mahdi'):.6f} --> very low since of the unknown word")


P('world' | 'the') = 0.0034117305458768875
P('are' | 'how') = 0.00017043656107715906
P('Alieu' | 'Mahdi') = 0.000025 --> very low since of the unknown word


In [9]:
def sentence_log_probability(words):
    """
    Calculate the log probability of a sentence using the bigram model. log P(W) sum of log P(w_i | w_i-1) for all words

    Higher (less negative) = more probable sentence
    Used to rank in real-word error correction
    """
    log_prob = 0.0
    prev = '<s>'
    for word in words:
        log_prob += math.log(bigram_probability(word, prev))
        prev = word
    return log_prob
# Testing the function by comparing correct vs incorrect sentence
sent_correct = ['the', 'stock', 'market', 'rose']
sent_wrong = ['the', 'stock', 'market', 'rows']

score_correct = sentence_log_probability(sent_correct)
score_wrong = sentence_log_probability(sent_wrong)

print(f"'the stock market rose -> score = {score_correct:.6f}")
print(f"'the stock market rows -> score = {score_wrong:.6f}")
print("Correct sentece score higher (less negative) = more probable sentence")

'the stock market rose -> score = -29.667057
'the stock market rows -> score = -29.667057
Correct sentece score higher (less negative) = more probable sentence


## Step 4: Edit Distance Calculation

The **Minimum Edit Distance (MED)** between two strings is calculated using **dynamic programming** (Wagner-Fischer algorithm).

This is the Damerau-Levenshtein distance with:
- Insert: cost 1
- Delete: cost 1
- Substitute: cost 1 (or cost 2 for Levenshtein variant)
- Transpose: cost 1

The edit distance matrix shows the optimal alignment between two strings.

In [10]:
def min_edit_distance(source, target, sub_cost=1):
    """
    Damerau-Levenshtein Edit Distance using dynamic programming.
    
    4 Operations and costs:
      - Insert: cost 1
      - Delete: cost 1
      - Substitute: cost sub_cost (1 for standard, 2 for Levenshtein)
      - Transpose: cost 1 (swap two adjacent characters)
    """
    n = len(source)
    m = len(target)
    
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if source[i-1] == target[j-1]:
                cost = 0
            else:
                cost = sub_cost
            
            dp[i][j] = min(
                dp[i-1][j] + 1,       # delete
                dp[i][j-1] + 1,       # insert
                dp[i-1][j-1] + cost   # substitute (or match)
            )
            
            # Transposition: swap two adjacent characters
            if i > 1 and j > 1 and source[i-1] == target[j-2] and source[i-2] == target[j-1]:
                dp[i][j] = min(dp[i][j], dp[i-2][j-2] + 1)
    
    return dp[n][m]

def print_edit_distance_matrix(source, target, sub_cost=1):
    """
    Print the full edit distance matrix for visualization
    """
    n, m = len(source), len(target)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if source[i-1] == target[j-1] else sub_cost
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
            
            # Transposition
            if i > 1 and j > 1 and source[i-1] == target[j-2] and source[i-2] == target[j-1]:
                dp[i][j] = min(dp[i][j], dp[i-2][j-2] + 1)
    
    header = '     ' + '  '.join([f'{c:>2}' for c in ' ' + target])
    print(header)
    for i in range(n + 1):
        label = ' ' if i == 0 else source[i-1]
        row = f'  {label}  ' + '  '.join([f'{dp[i][j]:>2}' for j in range(m + 1)])
        print(row)
    print(f"\nDamerau-Levenshtein Distance (sub_cost={sub_cost}): {dp[n][m]}")

In [11]:
# testing the function
print("Edit Distance Examples:")
pairs = [('stok', 'stock'), ('acress', 'actress'), ('teh', 'the'), ('graffe', 'giraffe')]
for s, t in pairs:
    d1 = min_edit_distance(s, t, sub_cost=1)
    d2 = min_edit_distance(s, t, sub_cost=2)
    print(f"\n'{s}' -> '{t}': MED(sub=1) = {d1}, Levenshtein(sub=2) = {d2}")

print()
print("Edit Distance Matrix: 'Mahdi' → 'Alieu'")
print("(Levenshtein variant: substitution cost = 2)")
print("===============================")
print_edit_distance_matrix('Mahdi', 'Alieu', sub_cost=2)

Edit Distance Examples:

'stok' -> 'stock': MED(sub=1) = 1, Levenshtein(sub=2) = 1

'acress' -> 'actress': MED(sub=1) = 1, Levenshtein(sub=2) = 1

'teh' -> 'the': MED(sub=1) = 1, Levenshtein(sub=2) = 1

'graffe' -> 'giraffe': MED(sub=1) = 1, Levenshtein(sub=2) = 1

Edit Distance Matrix: 'Mahdi' → 'Alieu'
(Levenshtein variant: substitution cost = 2)
          A   l   i   e   u
      0   1   2   3   4   5
  M   1   2   3   4   5   6
  a   2   3   4   5   6   7
  h   3   4   5   6   7   8
  d   4   5   6   7   8   9
  i   5   6   7   6   7   8

Damerau-Levenshtein Distance (sub_cost=2): 8


## Step 5: Edit Distance — Candidate Generation

Edit distance = minimum number of operations to transform one word into another.

**Damerau-Levenshtein: 4 operations**
1. **Delete** a letter: *acress* → *acess* (delete 'r')
2. **Insert** a letter: *acress* → *actress* (insert 't')
3. **Substitute** a letter: *acress* → *across* (replace 'e' with 'o')
4. **Transpose** two adjacent letters: *teh* → *the* (swap 'e' and 'h')


We generate all words that are 1 edit or 2 edits away from the input, then filter to keep only real dictionary words.

In [12]:
def edits_one(word):
    """
    Generate ALL possible strings that are exactly 1 edit away.
    Uses all 4 Damerau-Levenshtein operations.
    Returns raw candidates (including non-words).
    """
    letters = 'abcdefghijklmnopqrstuvwxyz'
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    
    deletes    = [L + R[1:]           for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces   = [L + c + R[1:]       for L, R in splits if R for c in letters]
    inserts    = [L + c + R           for L, R in splits for c in letters]
    
    return set(deletes + transposes + replaces + inserts)


def edits_two(word):
    """
    Generate all words that are exactly 2 edits away.
    Apply edits_one twice. Only keep words in vocabulary.
    """
    return set(
        e2 for e1 in edits_one(word)
           for e2 in edits_one(e1)
           if e2 in VOCABULARY
    )


# Test
test_word = 'acress'
all_edits = edits_one(test_word)
real_edits = all_edits & VOCABULARY
print(f"Word: '{test_word}'")
print(f"Total 1-edit candidates: {len(all_edits)}")
print(f"Real words among them: {len(real_edits)}")
print(f"Candidates: {sorted(real_edits)}")

Word: 'acress'
Total 1-edit candidates: 336
Real words among them: 5
Candidates: ['access', 'acres', 'across', 'actress', 'caress']


## Step 6: Confusion Matrices — Channel Model P(x|w)

The **Channel Model P(x|w)** answers: "If the correct word is `w`, how likely is the typo `x`?"

This is estimated from **confusion matrices** — tables that count how often each type of edit error occurs in real typo data.

**4 confusion matrices:**
- `del[x,y]` — count of times letter 'y' was deleted after letter 'x'
- `ins[x,y]` — count of times letter 'y' was inserted after letter 'x'
- `sub[x,y]` — count of times letter 'x' was typed instead of letter 'y'
- `trans[x,y]` — count of times letters 'x' and 'y' were transposed

**4 formulas for P(x|w)**:
- Deletion: P(x|w) = del[x,y] / count[xy]
- Insertion: P(x|w) = ins[x,y] / count[x]
- Substitution: P(x|w) = sub[x,y] / count[y]
- Transposition: P(x|w) = trans[x,y] / count[xy]

Where the denominator is always "how many opportunities existed to make this mistake".

In [20]:
def identify_edit(source, target):
    """
    Identify what single edit operation transforms source → target.
    Returns: (operation, char1, char2) or None if not a single edit.
    
    Operations:
      'del' — a character was deleted from target to get source
      'ins' — a character was inserted into source that's not in target
      'sub' — a character was substituted
      'trans' — two adjacent characters were transposed
    """
    ls, lt = len(source), len(target)
    
    # Deletion: source is 1 char shorter than target
    # e.g., source='acress', target='actress' → 't' was deleted after 'c'
    if ls == lt - 1:
        for i in range(lt):
            if source == target[:i] + target[i+1:]:
                prev_char = target[i-1] if i > 0 else '#'
                return ('del', prev_char, target[i])
    
    # Insertion: source is 1 char longer than target
    # e.g., source='acress', target='cress' → 'a' was inserted
    if ls == lt + 1:
        for i in range(ls):
            if target == source[:i] + source[i+1:]:
                prev_char = source[i-1] if i > 0 else '#'
                return ('ins', prev_char, source[i])
    
    # Substitution: same length, exactly 1 char different
    # e.g., source='acress', target='across' → 'e' substituted for 'o'
    if ls == lt:
        diffs = [(i, source[i], target[i]) for i in range(ls) if source[i] != target[i]]
        if len(diffs) == 1:
            i, s_char, t_char = diffs[0]
            return ('sub', s_char, t_char)
        
        # Transposition: same length, exactly 2 adjacent chars swapped
        if len(diffs) == 2:
            i, j = diffs[0][0], diffs[1][0]
            if j == i + 1 and source[i] == target[j] and source[j] == target[i]:
                return ('trans', source[i], source[j])
    
    return None


# Test
print("Edit identification examples:")
examples = [
    ('acress', 'actress'),  # deletion of 't' after 'c'
    ('acress', 'across'),   # substitution 'e' -> 'o'
    ('acress', 'cress'),    # insertion of 'a'
    ('teh', 'the'),         # transposition of 'e' and 'h'
]
for src, tgt in examples:
    edit = identify_edit(src, tgt)
    print(f"'{src}' -> '{tgt}': {edit}")

Edit identification examples:
'acress' -> 'actress': ('del', 'c', 't')
'acress' -> 'across': ('sub', 'e', 'o')
'acress' -> 'cress': ('ins', '#', 'a')
'teh' -> 'the': ('trans', 'e', 'h')


In [ ]:
# Build confusion matrices from REAL misspelling data

# Source: Peter Norvig's spell-errors.txt — a collection of real-world
# misspelling→correction pairs compiled from multiple sources including
# Wikipedia edits, Birkbeck corpus, and others. Contains ~7,840 lines
# covering thousands of real typo patterns.
#
# Format: correct_word: typo1, typo2, typo3*frequency, ...
#
# Reference: https://norvig.com/ngrams/spell-errors.txt

# --- Load real misspelling pairs from Norvig's spell-errors.txt ---
real_misspellings = []

with open('spell-errors.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if not line or ':' not in line:
            continue
        
        # Format: correct_word: typo1, typo2, typo3*5, ...
        correct, typos_str = line.split(':', 1)
        correct = correct.strip().lower()
        
        for typo in typos_str.split(','):
            typo = typo.strip()
            # Remove frequency marker (e.g., "fore*5" → "fore")
            if '*' in typo:
                typo = typo.split('*')[0].strip()
            if typo and typo != correct:
                real_misspellings.append((typo, correct))

print(f"Loaded {len(real_misspellings)} real misspelling pairs from Norvig's spell-errors.txt")
print(f"First 10 pairs: {real_misspellings[:10]}")

# Count character bigrams and unigrams from the corpus 
# These are the DENOMINATORS in the channel model formulas (Slide 19):
#   - char_bigram_counts['th'] = how many times 'th' appears in the corpus
#   - char_unigram_counts['t'] = how many times 't' appears in the corpus

char_bigram_counts = Counter()
char_unigram_counts = Counter()

for word in corpus_words:
    for ch in word:
        char_unigram_counts[ch] += 1
    for i in range(len(word) - 1):
        char_bigram_counts[word[i] + word[i+1]] += 1

# Initialize confusion matrices with Laplace smoothing (add-1)
del_matrix = defaultdict(lambda: defaultdict(lambda: 1))   # del[x][y]
ins_matrix = defaultdict(lambda: defaultdict(lambda: 1))   # ins[x][y]
sub_matrix = defaultdict(lambda: defaultdict(lambda: 1))   # sub[x][y]
trans_matrix = defaultdict(lambda: defaultdict(lambda: 1)) # trans[x][y]

# Extract edit operations from real misspelling pairs
edit_counts = {'del': 0, 'ins': 0, 'sub': 0, 'trans': 0, 'multi': 0}

for typo, correct in real_misspellings:
    edit = identify_edit(typo, correct)
    
    if edit is None:
        edit_counts['multi'] += 1
        continue  # skip multi-edit pairs (can't attribute to single matrix)
    
    op, c1, c2 = edit
    edit_counts[op] += 1
    
    if op == 'del':
        del_matrix[c1][c2] += 10  # weight each real observation
    elif op == 'ins':
        ins_matrix[c1][c2] += 10
    elif op == 'sub':
        sub_matrix[c1][c2] += 10
    elif op == 'trans':
        trans_matrix[c1][c2] += 10

print(f"\nEdit operations extracted from real data:")
print(f"  Deletions:      {edit_counts['del']}")
print(f"  Insertions:     {edit_counts['ins']}")
print(f"  Substitutions:  {edit_counts['sub']}")
print(f"  Transpositions: {edit_counts['trans']}")
print(f"  Multi-edit:     {edit_counts['multi']} (skipped — not single-edit)")
print(f"  Total single-edit: {sum(v for k,v in edit_counts.items() if k != 'multi')}")

# --- Show top entries in each matrix ---
print(f"\nCharacter unigrams tracked: {len(char_unigram_counts)}")
print(f"Character bigrams tracked: {len(char_bigram_counts)}")

print(f"\nTop substitution confusions (from real misspellings):")
sub_pairs = []
for c1 in sub_matrix:
    for c2 in sub_matrix[c1]:
        if sub_matrix[c1][c2] > 1:
            sub_pairs.append((c1, c2, sub_matrix[c1][c2]))
sub_pairs.sort(key=lambda x: x[2], reverse=True)
for c1, c2, count in sub_pairs[:10]:
    print(f"  '{c1}' typed instead of '{c2}': count = {count}")

print(f"\nTop transposition confusions (from real misspellings):")
trans_pairs = []
for c1 in trans_matrix:
    for c2 in trans_matrix[c1]:
        if trans_matrix[c1][c2] > 1:
            trans_pairs.append((c1, c2, trans_matrix[c1][c2]))
trans_pairs.sort(key=lambda x: x[2], reverse=True)
for c1, c2, count in trans_pairs[:10]:
    print(f"  '{c1}' and '{c2}' swapped: count = {count}")

Loaded 39684 real misspelling pairs from Norvig's spell-errors.txt
First 10 pairs: [('rainning', 'raining'), ('raning', 'raining'), ('writtings', 'writings'), ('disparingly', 'disparagingly'), ('yello', 'yellow'), ('forer', 'four'), ('fours', 'four'), ('fuore', 'four'), ('fore', 'four'), ('for', 'four')]

Edit operations extracted from real data:
  Deletions:      4677
  Insertions:     3233
  Substitutions:  5127
  Transpositions: 1029
  Multi-edit:     25618 (skipped — not single-edit)
  Total single-edit: 14066

Character unigrams tracked: 26
Character bigrams tracked: 585

Top substitution confusions (from real misspellings):
  'a' typed instead of 'e': count = 4171
  'e' typed instead of 'i': count = 3651
  'e' typed instead of 'a': count = 3351
  'i' typed instead of 'e': count = 2831
  'a' typed instead of 'i': count = 1831
  'a' typed instead of 'o': count = 1721
  'o' typed instead of 'a': count = 1621
  's' typed instead of 'c': count = 1601
  'e' typed instead of 'o': count 

In [21]:
import pandas as pd

# Pick which matrix to view
matrix = sub_matrix  # change to del_matrix, ins_matrix, trans_matrix
name = "Substitution"

# Get all characters that appear in the matrix
rows = sorted(matrix.keys())
cols = sorted(set(c for r in matrix for c in matrix[r]))

# Build a table
data = {}
for c in cols:
    data[c] = [matrix[r][c] for r in rows]

df = pd.DataFrame(data, index=rows)
print(f"\n{name} Matrix (rows = what was typed, cols = what was intended):")
print(df.to_string())


Substitution Matrix (rows = what was typed, cols = what was intended):
    '    -   .   _     a    b     c    d     e    f    g    h     i    j    k    l     m    n     o    p   q    r     s    t    u    v    w   x    y    z
'   1    1   1   1     1    1     1    1    31    1    1    1     1    1    1    1     1    1     1    1   1    1     1    1    1    1    1   1    1    1
-  11    1   1  71     1    1     1    1    11    1    1    1     1    1    1   11     1    1     1    1   1    1     1   11    1    1    1   1   11    1
A   1    1   1   1    21    1     1    1    11    1    1    1     1    1    1    1     1    1     1    1   1    1     1    1    1    1    1   1    1    1
B   1    1   1   1     1   41     1    1     1    1    1    1     1    1    1    1    11    1     1    1   1    1     1    1    1    1    1   1    1    1
C   1    1   1   1     1    1    41    1     1    1    1    1     1    1    1    1     1    1     1    1   1    1     1    1    1    1    1   1    1    1
D   

In [15]:
def channel_probability(typo, candidate):
    """
    Channel Model: P(x|w) — probability of making typo x given correct word w.
    
    Uses the confusion matrices and the formulas from Slide 19:
      Deletion:      P(x|w) = del[prev, deleted_char] / count[prev + deleted_char]
      Insertion:     P(x|w) = ins[prev, inserted_char] / count[prev]
      Substitution:  P(x|w) = sub[typed_char, intended_char] / count[intended_char]
      Transposition: P(x|w) = trans[char1, char2] / count[char1 + char2]
    
    If the edit can't be identified (multi-edit), falls back to edit distance penalty.
    """
    edit = identify_edit(typo, candidate)
    
    if edit is None:
        # Multi-edit: use exponential decay based on edit distance
        dist = min_edit_distance(typo, candidate)
        return math.exp(-2 * dist)
    
    op, c1, c2 = edit
    
    if op == 'del':
        # Character c2 was deleted after c1
        bigram = c1 + c2
        numerator = del_matrix[c1][c2]
        denominator = char_bigram_counts.get(bigram, 1) + 26  # smoothed
        return numerator / denominator
    
    elif op == 'ins':
        # Character c2 was inserted after c1
        numerator = ins_matrix[c1][c2]
        denominator = char_unigram_counts.get(c1, 1) + 26  # smoothed
        return numerator / denominator
    
    elif op == 'sub':
        # Character c1 was typed instead of c2
        numerator = sub_matrix[c1][c2]
        denominator = char_unigram_counts.get(c2, 1) + 26  # smoothed
        return numerator / denominator
    
    elif op == 'trans':
        # Characters c1 and c2 were transposed
        bigram = c1 + c2
        numerator = trans_matrix[c1][c2]
        denominator = char_bigram_counts.get(bigram, 1) + 26  # smoothed
        return numerator / denominator
    
    return 1e-6  # fallback


# Test: channel probabilities for 'acress' candidates
typo = 'acress'
candidates = ['actress', 'across', 'access', 'cress', 'caress']
print(f"Channel Model P(x|w) for typo '{typo}':")
print(f"{'Candidate':<12} {'Edit':<35} {'P(x|w)':<12}")
print("-" * 60)
for c in candidates:
    edit = identify_edit(typo, c)
    prob = channel_probability(typo, c)
    print(f"{c:<12} {str(edit):<35} {prob:.6f}")

Channel Model P(x|w) for typo 'acress':
Candidate    Edit                                P(x|w)      
------------------------------------------------------------
actress      ('del', 'c', 't')                   0.005790
across       ('sub', 'e', 'o')                   0.003773
access       ('sub', 'r', 'c')                   0.000218
cress        ('ins', '#', 'a')                   3.000000
caress       ('trans', 'a', 'c')                 0.002541


## Step 7: Non-Word Error Correction (Noisy Channel)

**Non-word error** = the word is NOT in the vocabulary (e.g., *acress*, *stok*)

**Algorithm:**
1. **Detect**: Check if word is in the dictionary → if NOT, it's a non-word error
2. **Generate candidates**: Find all real words within edit distance 1-2
3. **Score each candidate**: Score(w) = P(x|w) × P(w)
   - P(x|w) = channel probability from confusion matrices
   - P(w) = prior probability from word frequencies
4. **Select best**: Pick the candidate with the highest score

This is the **Noisy Channel Model**: ŵ = argmax P(x|w) × P(w)

In [23]:
def get_candidates(word):
    """
    Get correction candidates for a misspelled word.
    Priority (Slide 16 — 80% of errors are edit distance 1):
      1. The word itself (if already correct)
      2. Words 1 edit away
      3. Words 2 edits away (if no 1-edit candidates)
      4. The original word (fallback)
    """
    if word in VOCABULARY:
        return {word}
    
    candidates_1 = edits_one(word) & VOCABULARY
    if candidates_1:
        return candidates_1
    
    candidates_2 = edits_two(word)
    if candidates_2:
        return candidates_2
    
    return {word}


def correct_nonword(word):
    """
    Correct a single non-word error using the Noisy Channel Model.
    
    Score(w) = P(x|w) × P(w)
    
    Returns: (corrected_word, candidates_with_scores)
    """
    word = word.lower()
    
    if word in VOCABULARY:
        return word, []
    
    candidates = get_candidates(word)
    
    # Score each candidate using the Noisy Channel formula
    scored = []
    for c in candidates:
        # P(x|w) — channel model
        p_xw = channel_probability(word, c)

        # P(w)   — language model (prior)
        p_w  = word_probability(c)            

        # Noisy channel score
        score = p_xw * p_w

        scored.append((c, p_xw, p_w, score))
    
    # Sort by score (highest first)
    scored.sort(key=lambda x: x[3], reverse=True)
    
    best = scored[0][0] if scored else word
    return best, scored[:5]


# Test with the 'acress' example from Slide 20
print("=" * 75)
print("Non-Word Correction using Noisy Channel Model")
print("=" * 75)

test_words = ['acress', 'stok', 'teh', 'mney', 'compny', 'bnk']
for w in test_words:
    corrected, scores = correct_nonword(w)
    print(f"\nTypo: '{w}' -> Correction: '{corrected}'")
    if scores:
        print(f"  {'Candidate':<12} {'P(x|w)':<12} {'P(w)':<14} {'Score':<14}")
        print(f"  {'-'*52}")
        for c, pxw, pw, score in scores[:5]:
            print(f"  {c:<12} {pxw:<12.6f} {pw:<14.8e} {score:<14.8e}")

Non-Word Correction using Noisy Channel Model

Typo: 'acress' -> Correction: 'across'
  Candidate    P(x|w)       P(w)           Score         
  ----------------------------------------------------
  across       0.003773     2.76921572e-04 1.04485872e-06
  actress      0.005790     6.84965018e-06 3.96612813e-08
  acres        0.000833     4.40334654e-05 3.66742858e-08
  access       0.000218     2.44630364e-05 5.32305341e-09
  caress       0.002541     1.95704291e-06 4.97234673e-09

Typo: 'stok' -> Correction: 'stock'
  Candidate    P(x|w)       P(w)           Score         
  ----------------------------------------------------
  stock        0.040896     1.44821175e-04 5.92255361e-06
  stop         0.000333     1.18401096e-04 3.94568496e-08

Typo: 'teh' -> Correction: 'the'
  Candidate    P(x|w)       P(w)           Score         
  ----------------------------------------------------
  the          0.055375     6.84691032e-02 3.79144871e-03
  eh           1.888889     8.80669309e-

## Step 8: Real-Word Error Correction

**Real-word error** = the word IS in the dictionary but is wrong in context.

Example: "two of **thew**" -> "thew" is a real English word but wrong here -> should be "the"

**Algorithm (one-error-per-sentence simplification):**
1. For EVERY word in the sentence, generate candidates (edit distance 1 + the word itself)
2. Assume only ONE word is wrong at a time
3. Try replacing each word with each of its candidates
4. Score the resulting sentence using bigram probability: P(W) = Π P(wᵢ | wᵢ₋₁)
5. Pick the sentence with the highest score: W* = argmax P(W)

**Key difference from non-word correction:**
- Non-word: score individual words with P(x|w) × P(w)
- Real-word: score entire sentences with P(W) using bigrams

In [24]:
def correct_realword(sentence):
    """
    Real-word error correction using the one-error-per-sentence approach
    
    For each word position:
      1. Generate candidates = {word itself} ∪ {edit distance 1 words in vocabulary}
      2. Try replacing just this one word with each candidate
      3. Score the full sentence with bigram probabilities
      4. Keep the best-scoring replacement
    
    Returns: (corrected_sentence, list_of_corrections)
    """
    words = sentence.lower().split()
    
    # Score the original sentence
    original_score = sentence_log_probability(words)
    
    best_sentence = list(words)
    best_score = original_score
    corrections = []
    
    # Try replacing ONE word at a time (one-error-per-sentence)
    for i, word in enumerate(words):
        # Only consider words that are in the vocabulary (real-word errors)
        if word not in VOCABULARY:
            continue
        
        # Generate candidates: the word itself + 1-edit words in vocabulary
        candidates = (edits_one(word) & VOCABULARY) | {word}
        
        for candidate in candidates:
            if candidate == word:
                continue  # skip the original word
            
            # Create test sentence with this candidate
            test_words = list(words)
            test_words[i] = candidate
            
            # Score the test sentence using bigram model
            test_score = sentence_log_probability(test_words)
            
            # If this is better than the current best, update
            if test_score > best_score:
                best_sentence = list(test_words)
                best_score = test_score
                corrections = [(word, candidate, i)]
    
    return ' '.join(best_sentence), corrections


# Test real-word correction
print("=" * 65)
print("Real-Word Error Correction (One-Error-Per-Sentence)")
print("=" * 65)

test_sentences = [
    "the stock market rows today",
    "oil prices fell an tuesday",
    "the company made a beg profit",
]

for sent in test_sentences:
    corrected, corrections = correct_realword(sent)
    print(f"\n  Input:     {sent}")
    print(f"  Corrected: {corrected}")
    
    orig_score = sentence_log_probability(sent.lower().split())
    corr_score = sentence_log_probability(corrected.split())
    print(f"  Score:     {orig_score:.4f} -> {corr_score:.4f}")
    
    if corrections:
        for orig, fix, pos in corrections:
            print(f"  Changed:   '{orig}' -> '{fix}' at position {pos}")
    else:
        print(f"  No changes needed")

Real-Word Error Correction (One-Error-Per-Sentence)

  Input:     the stock market rows today
  Corrected: the stock market lows today
  Score:     -40.2700 -> -40.2696
  Changed:   'rows' -> 'lows' at position 3

  Input:     oil prices fell an tuesday
  Corrected: oil prices fell on tuesday
  Score:     -53.9877 -> -49.5429
  Changed:   'an' -> 'on' at position 3

  Input:     the company made a beg profit
  Corrected: the company made a big profit
  Score:     -47.5608 -> -43.1386
  Changed:   'beg' -> 'big' at position 4


## Step 9: Combined Spelling Correction System

The complete system runs **two passes** (as described in the slides):

**Pass 1 — Non-Word Errors:** Check each word against the dictionary.
If a word is NOT found, correct it using the Noisy Channel: ŵ = argmax P(x|w) × P(w)

**Pass 2 — Real-Word Errors:** After fixing non-words, check context.
For each word, try replacing it and score the whole sentence: W* = argmax P(W)

In [30]:
def spell_correct(sentence):
    """
    Full spelling correction pipeline:
      Pass 1: Fix non-word errors using Noisy Channel (P(x|w) × P(w))
      Pass 2: Fix real-word errors using bigram sentence scoring (P(W))
    
    Returns: (corrected_sentence, list_of_all_corrections)
    """
    words = sentence.lower().split()
    all_corrections = []
    
    # === PASS 1: Non-word error correction ===
    corrected_words = []
    for word in words:
        if word not in VOCABULARY:
            fixed, scores = correct_nonword(word)
            if fixed != word:
                all_corrections.append({
                    'type': 'non-word',
                    'original': word,
                    'corrected': fixed,
                    'edit_distance': min_edit_distance(word, fixed)
                })
            corrected_words.append(fixed)
        else:
            corrected_words.append(word)
    
    after_pass1 = ' '.join(corrected_words)
    
    # === PASS 2: Real-word error correction ===
    final_sentence, realword_corrections = correct_realword(after_pass1)
    for orig, fix, pos in realword_corrections:
        all_corrections.append({
            'type': 'real-word',
            'original': orig,
            'corrected': fix,
            'edit_distance': min_edit_distance(orig, fix)
        })
    
    return final_sentence, all_corrections


# Test the full system
test_sentences = [
    "the stok market rows today",
    "oil pricess fell on tusday",
    "the compny reported beg losses",
    "teh goverment announced new polices",
    "investors ar worried about inflaton",
]

print("==================================================")
print("FULL SPELLING CORRECTION SYSTEM")
print("Pass 1: Non-word errors (Noisy Channel)")
print("Pass 2: Real-word errors (Bigram sentence scoring)")
print("==================================================")

for sent in test_sentences:
    corrected, corrections = spell_correct(sent)
    print(f"\n  Input:     {sent}")
    print(f"  Output:    {corrected}")
    if corrections:
        for c in corrections:
            print(f"    [{c['type']:>9}] '{c['original']}' -> '{c['corrected']}' (edit distance: {c['edit_distance']})")
    else:
        print(f"    No errors found")
    print("----------------------------------------------------" )

FULL SPELLING CORRECTION SYSTEM
Pass 1: Non-word errors (Noisy Channel)
Pass 2: Real-word errors (Bigram sentence scoring)

  Input:     the stok market rows today
  Output:    the stock market lows today
    [ non-word] 'stok' -> 'stock' (edit distance: 1)
    [real-word] 'rows' -> 'lows' (edit distance: 1)
----------------------------------------------------

  Input:     oil pricess fell on tusday
  Output:    soil process fell on tuesday
    [ non-word] 'pricess' -> 'process' (edit distance: 1)
    [ non-word] 'tusday' -> 'tuesday' (edit distance: 1)
    [real-word] 'oil' -> 'soil' (edit distance: 1)
----------------------------------------------------

  Input:     the compny reported beg losses
  Output:    the company reported be losses
    [ non-word] 'compny' -> 'company' (edit distance: 1)
    [real-word] 'beg' -> 'be' (edit distance: 1)
----------------------------------------------------

  Input:     teh goverment announced new polices
  Output:    the government announced

## Step 10: Evaluation

In [19]:
# Non-word error correction evaluation
nonword_test_cases = [
    ('mney', 'money'),
    ('stok', 'stock'),
    ('teh', 'the'),
    ('bnk', 'bank'),
    ('compny', 'company'),
    ('goverment', 'government'),
    ('pricess', 'prices'),
    ('anounced', 'announced'),
    ('investrs', 'investors'),
    ('reportd', 'reported'),
    ('finncial', 'financial'),
    ('revnue', 'revenue'),
    ('strng', 'strong'),
    ('growtth', 'growth'),
    ('ecnomic', 'economic'),
]

correct_count = 0
print("Non-Word Error Correction Evaluation (Noisy Channel Model)")
print("=" * 70)
print(f"{'Misspelled':<15} {'Expected':<15} {'Got':<15} {'Edit Dist':<10} {'Result'}")
print("-" * 70)

for misspelled, expected in nonword_test_cases:
    got, scores = correct_nonword(misspelled)
    dist = min_edit_distance(misspelled, got)
    is_correct = got == expected
    if is_correct:
        correct_count += 1
    status = '✓' if is_correct else '✗'
    print(f"{misspelled:<15} {expected:<15} {got:<15} {dist:<10} {status}")

accuracy = correct_count / len(nonword_test_cases) * 100
print("-" * 70)
print(f"Accuracy: {correct_count}/{len(nonword_test_cases)} = {accuracy:.1f}%")

Non-Word Error Correction Evaluation (Noisy Channel Model)
Misspelled      Expected        Got             Edit Dist  Result
----------------------------------------------------------------------
mney            money           money           1          ✓
stok            stock           stock           1          ✓
teh             the             the             1          ✓
bnk             bank            bank            1          ✓
compny          company         company         1          ✓
goverment       government      government      1          ✓
pricess         prices          process         1          ✗
anounced        announced       announced       1          ✓
investrs        investors       investors       1          ✓
reportd         reported        reported        1          ✓
finncial        financial       financial       1          ✓
revnue          revenue         revenue         1          ✓
strng           strong          strong          1          ✓
growtth    